# 🏦 Bank Statement PDF to CSV Converter - Interactive Testing

This notebook lets you test and analyze the PDF conversion accuracy.

**What you can do:**
- Test conversion with sample PDFs
- View extracted transactions in DataFrame
- Analyze accuracy metrics
- Preview CSV formatting
- Test different bank formats

## 1️⃣ Setup & Imports

In [ ]:
import sys
import os
import pandas as pd
from pathlib import Path

# Add project directory to path
project_dir = r"C:\Users\gadip\OneDrive\Documents\MERN\bankstatement converter"
sys.path.insert(0, project_dir)
os.chdir(project_dir)

# Import processor functions
from processor import (
    convert_pdf_to_csv,
    is_pdf_text_based,
    extract_text_pdf_enhanced,
    extract_text_ocr,
    parse_bank_statement_to_rows
)

print("✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")

## 2️⃣ List Available PDFs

In [ ]:
# Find all PDFs in uploads folder
uploads_dir = Path("uploads")
if uploads_dir.exists():
    pdf_files = list(uploads_dir.glob("*.pdf"))
    print(f"📄 Found {len(pdf_files)} PDF files:\n")
    for i, pdf in enumerate(pdf_files, 1):
        size_kb = pdf.stat().st_size / 1024
        print(f"{i}. {pdf.name} ({size_kb:.1f} KB)")
else:
    print("⚠️  No uploads folder found. Create one and add test PDFs.")
    pdf_files = []

## 3️⃣ Test PDF Conversion

**Choose a PDF to test:**

In [ ]:
# Select PDF to test (change index as needed)
if pdf_files:
    test_pdf = pdf_files[0]  # Change index to test different PDFs
    print(f"🧪 Testing: {test_pdf.name}")
    print(f"📊 Size: {test_pdf.stat().st_size / 1024:.1f} KB")
    
    # Check if scanned or text-based
    is_text_based = is_pdf_text_based(str(test_pdf))
    pdf_type = "Text-based" if is_text_based else "Scanned (OCR required)"
    print(f"📝 Type: {pdf_type}")
else:
    print("❌ No PDF files found. Please add PDFs to the uploads folder.")

## 4️⃣ Extract Text & View Sample

In [ ]:
if pdf_files:
    print("🔄 Extracting text from PDF...\n")
    
    if is_text_based:
        text, tables = extract_text_pdf_enhanced(str(test_pdf))
        print(f"✅ Extracted {len(text)} characters")
        print(f"📊 Found {len(tables) if tables else 0} tables\n")
    else:
        text = extract_text_ocr(str(test_pdf))
        tables = None
        print(f"✅ OCR extracted {len(text)} characters\n")
    
    print("📄 First 500 characters of extracted text:")
    print("=" * 60)
    print(text[:500])
    print("=" * 60)

## 5️⃣ Parse Transactions & Create DataFrame

In [ ]:
if pdf_files:
    print("🔄 Parsing transactions...\n")
    
    # Parse transactions
    rows = parse_bank_statement_to_rows(text, tables)
    
    print(f"✅ Extracted {len(rows)} transactions\n")
    
    # Create DataFrame
    if rows:
        df = pd.DataFrame(rows)
        
        # Display info
        print("📊 DataFrame Info:")
        print(f"   - Rows: {len(df)}")
        print(f"   - Columns: {list(df.columns)}")
        print(f"   - Date range: {df['date'].min()} to {df['date'].max()}")
        
        # Show sample
        print("\n📋 Sample Transactions (first 10):")
        display(df.head(10))
    else:
        print("⚠️  No transactions extracted!")

## 6️⃣ Analyze Transaction Types

In [ ]:
if pdf_files and rows:
    # Detect transaction types
    def detect_transaction_type(description):
        desc_lower = description.lower()
        if any(word in desc_lower for word in ['check', 'cheque']):
            return 'Check'
        elif any(word in desc_lower for word in ['atm', 'withdrawal']):
            return 'ATM Withdrawal'
        elif any(word in desc_lower for word in ['deposit', 'credit', 'payroll']):
            return 'Deposit/Credit'
        elif 'pos' in desc_lower or 'purchase' in desc_lower:
            return 'POS Purchase'
        elif any(word in desc_lower for word in ['fee', 'charge']):
            return 'Fee/Charge'
        elif 'interest' in desc_lower:
            return 'Interest'
        else:
            return 'Other'
    
    df['transaction_type'] = df['description'].apply(detect_transaction_type)
    
    # Analyze transaction types
    print("📊 Transaction Type Breakdown:\n")
    type_counts = df['transaction_type'].value_counts()
    for trans_type, count in type_counts.items():
        percentage = (count / len(df)) * 100
        print(f"   {trans_type:20s}: {count:3d} ({percentage:5.1f}%)")
    
    print("\n📈 Transaction Type Distribution:")
    display(type_counts.plot(kind='bar', figsize=(10, 5), title='Transaction Types'))

## 7️⃣ Analyze Amounts & Statistics

In [ ]:
if pdf_files and rows:
    # Convert amount to numeric
    df['amount_numeric'] = pd.to_numeric(df['amount'], errors='coerce')
    
    # Calculate statistics
    print("💰 Amount Statistics:\n")
    print(f"   Total Transactions: {len(df)}")
    print(f"   Valid Amounts: {df['amount_numeric'].notna().sum()}")
    print(f"   Invalid Amounts: {df['amount_numeric'].isna().sum()}")
    print(f"\n   Min Amount: ${df['amount_numeric'].min():,.2f}")
    print(f"   Max Amount: ${df['amount_numeric'].max():,.2f}")
    print(f"   Mean Amount: ${df['amount_numeric'].mean():,.2f}")
    print(f"   Median Amount: ${df['amount_numeric'].median():,.2f}")
    print(f"   Total Sum: ${df['amount_numeric'].sum():,.2f}")
    
    # Debits vs Credits
    debits = df[df['amount_numeric'] < 0]['amount_numeric'].sum()
    credits = df[df['amount_numeric'] > 0]['amount_numeric'].sum()
    
    print(f"\n   Total Debits: ${abs(debits):,.2f}")
    print(f"   Total Credits: ${credits:,.2f}")
    print(f"   Net Change: ${credits + debits:,.2f}")

## 8️⃣ Preview CSV Output Format

In [ ]:
if pdf_files and rows:
    print("📄 CSV Output Preview (first 500 characters):\n")
    print("=" * 60)
    
    # Convert to CSV string
    csv_preview = df[['date', 'description', 'amount']].head(10).to_csv(index=False)
    print(csv_preview[:500])
    print("=" * 60)
    
    # Show CSV format options
    print("\n💡 CSV Format Options:\n")
    print("Current format:")
    print("  - Date,Description,Amount")
    print("  - Simple 3-column layout")
    print("\nEnhanced format could include:")
    print("  - Transaction Type column")
    print("  - Debit/Credit columns (split amounts)")
    print("  - Balance column")
    print("  - Category column")
    print("  - Notes/Memo column")

## 9️⃣ Enhanced CSV Format Example

In [ ]:
if pdf_files and rows:
    # Create enhanced format
    df_enhanced = df.copy()
    
    # Add debit/credit columns
    df_enhanced['debit'] = df_enhanced['amount_numeric'].apply(
        lambda x: f"{abs(x):.2f}" if x < 0 else ""
    )
    df_enhanced['credit'] = df_enhanced['amount_numeric'].apply(
        lambda x: f"{x:.2f}" if x > 0 else ""
    )
    
    # Add balance column (cumulative sum)
    df_enhanced['balance'] = df_enhanced['amount_numeric'].cumsum()
    df_enhanced['balance'] = df_enhanced['balance'].apply(lambda x: f"{x:,.2f}")
    
    # Select columns for enhanced CSV
    df_enhanced_output = df_enhanced[[
        'date', 'description', 'transaction_type', 'debit', 'credit', 'balance'
    ]]
    
    print("✨ Enhanced CSV Format Preview:\n")
    display(df_enhanced_output.head(10))
    
    print("\n📊 Enhanced format includes:")
    print("  ✅ Date (YYYY-MM-DD)")
    print("  ✅ Description")
    print("  ✅ Transaction Type")
    print("  ✅ Debit (negative amounts)")
    print("  ✅ Credit (positive amounts)")
    print("  ✅ Running Balance")

## 🔟 Accuracy Assessment

In [ ]:
if pdf_files and rows:
    print("🎯 Accuracy Assessment:\n")
    
    # Check for missing data
    missing_dates = df['date'].isna().sum()
    missing_descriptions = df['description'].isna().sum()
    missing_amounts = df['amount'].isna().sum()
    
    total_rows = len(df)
    
    print("📊 Data Completeness:")
    print(f"   Total Transactions: {total_rows}")
    print(f"   Missing Dates: {missing_dates} ({missing_dates/total_rows*100:.1f}%)")
    print(f"   Missing Descriptions: {missing_descriptions} ({missing_descriptions/total_rows*100:.1f}%)")
    print(f"   Missing Amounts: {missing_amounts} ({missing_amounts/total_rows*100:.1f}%)")
    
    # Calculate accuracy score
    complete_rows = total_rows - max(missing_dates, missing_descriptions, missing_amounts)
    accuracy_score = (complete_rows / total_rows) * 100
    
    print(f"\n✅ Complete Rows: {complete_rows}/{total_rows}")
    print(f"🎯 Accuracy Score: {accuracy_score:.1f}%")
    
    if accuracy_score >= 95:
        print("\n🌟 Excellent! Accuracy is 95% or higher.")
    elif accuracy_score >= 90:
        print("\n👍 Good! Accuracy is above 90%.")
    elif accuracy_score >= 80:
        print("\n⚠️  Fair. Some improvements needed.")
    else:
        print("\n❌ Poor accuracy. Significant improvements needed.")

## 1️⃣1️⃣ Full Conversion Test

In [ ]:
if pdf_files:
    print("🔄 Running full conversion (PDF → CSV)...\n")
    
    try:
        csv_path = convert_pdf_to_csv(str(test_pdf))
        print(f"✅ Conversion successful!")
        print(f"📁 CSV saved to: {csv_path}")
        
        # Read and display CSV
        csv_df = pd.read_csv(csv_path)
        print(f"\n📊 CSV contains {len(csv_df)} rows")
        print("\n📋 First 10 rows from CSV file:")
        display(csv_df.head(10))
        
        # Show file size
        csv_size = os.path.getsize(csv_path) / 1024
        print(f"\n💾 CSV file size: {csv_size:.1f} KB")
        
    except Exception as e:
        print(f"❌ Conversion failed: {str(e)}")

## 1️⃣2️⃣ Export Enhanced CSV

In [ ]:
if pdf_files and rows:
    # Save enhanced CSV
    output_path = "enhanced_transactions.csv"
    df_enhanced_output.to_csv(output_path, index=False)
    
    print(f"✅ Enhanced CSV exported to: {output_path}")
    print(f"💾 File size: {os.path.getsize(output_path) / 1024:.1f} KB")
    print(f"📊 Contains {len(df_enhanced_output)} transactions")
    
    print("\n✨ Enhanced format includes:")
    print("   - Standardized dates (YYYY-MM-DD)")
    print("   - Transaction types (auto-detected)")
    print("   - Separate Debit/Credit columns")
    print("   - Running balance calculation")
    print("   - Clean, professional formatting")

## 📝 Summary & Next Steps

**What you tested:**
- ✅ PDF text extraction (OCR or direct)
- ✅ Transaction parsing
- ✅ Data accuracy assessment
- ✅ CSV format options
- ✅ Enhanced CSV generation

**To improve accuracy further:**
1. Test with multiple bank statement formats
2. Adjust regex patterns for your specific bank
3. Add custom date format parsing
4. Implement balance validation
5. Add transaction categorization

**Next steps:**
- Upload your actual bank statements to test
- Review any parsing errors
- Customize the CSV format as needed
- Integrate enhanced format into the web app